# ComfyUI＋LTX-2.3 個人テスト用ノートブック

上のセルから順に、左の ▶ ボタンで実行します。

- **毎回実行**：【準備】【ComfyUIの導入】【起動と見張り】
- **初回だけ（2回目以降は自動でスキップ）**：【モデルのダウンロード】

このノートブックは外部へデータを送らない設定（`--offline`）でComfyUIを起動します。
画面はColab標準の表示機能で開き、Cloudflareなどの中継サービスは使いません。

In [ ]:
# 【準備】GPUの確認・Google Driveの接続・保存先フォルダの作成
import os, subprocess

print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout or 'GPUが見つかりません。「ランタイム」→「ランタイムのタイプを変更」でGPUを選んでください')

from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/ComfyUI_LTX'   # Drive上の保存先
MODELS = f'{BASE}/models'                     # モデル（約43GB）
OUTPUT = f'{BASE}/output'                     # 作った動画
USERDIR = f'{BASE}/user'                      # ComfyUIの設定・保存したワークフロー

for sub in ['checkpoints', 'loras', 'text_encoders', 'latent_upscale_models']:
    os.makedirs(f'{MODELS}/{sub}', exist_ok=True)
for d in [OUTPUT, USERDIR]:
    os.makedirs(d, exist_ok=True)

IDLE_MINUTES = 30   # 生成していない時間がこの分数続いたら自動で切断
MAX_HOURS = 4       # 起動からこの時間が過ぎたら自動で切断

print('保存先:', BASE)

In [ ]:
# 【モデルのダウンロード】初回だけ時間がかかります（約43GB）。2回目以降は自動でスキップします
# 途中で止まった場合は、このセルをもう一度実行すると続きから再開します
import os, requests
from IPython import get_ipython

# ComfyUI公式テンプレート「LTX-2.3 Text to Video」が指定しているファイル
MODEL_FILES = [
    ('checkpoints',           'https://huggingface.co/Lightricks/LTX-2.3-fp8/resolve/main/ltx-2.3-22b-dev-fp8.safetensors'),
    ('loras',                 'https://huggingface.co/Comfy-Org/ltx-2.3/resolve/main/split_files/loras/ltx_2.3_22b_distilled_1.1_lora_dynamic_fro09_avg_rank_111_bf16.safetensors'),
    ('loras',                 'https://huggingface.co/Comfy-Org/ltx-2/resolve/main/split_files/loras/gemma-3-12b-it-abliterated_lora_rank64_bf16.safetensors'),
    ('text_encoders',         'https://huggingface.co/Comfy-Org/ltx-2/resolve/main/split_files/text_encoders/gemma_3_12B_it_fp4_mixed.safetensors'),
    ('latent_upscale_models', 'https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-spatial-upscaler-x2-1.1.safetensors'),
]

def expected_size(url):
    try:
        r = requests.head(url, allow_redirects=True, timeout=60)
        return int(r.headers.get('content-length', 0))
    except Exception:
        return 0

def is_done(path, size):
    if not os.path.exists(path):
        return False
    return os.path.getsize(path) == size if size else os.path.getsize(path) > 1_000_000

all_ok = True
for sub, url in MODEL_FILES:
    name = url.split('/')[-1]
    dest = f'{MODELS}/{sub}/{name}'
    size = expected_size(url)
    if is_done(dest, size):
        print(f'済み: {name}')
        continue
    print(f'ダウンロード中: {name}（{size / 1e9:.1f} GB）')
    get_ipython().system(f'wget -c -q --show-progress -O "{dest}" "{url}"')
    ok = is_done(dest, size)
    all_ok = all_ok and ok
    print('  → 完了' if ok else '  → 未完了：このセルをもう一度実行してください')

print('\nすべてのモデルがそろいました' if all_ok else '\n未完了のファイルがあります')

In [ ]:
# 【ComfyUIの導入】毎回実行（3〜5分）。Colabは終了すると中身が消えるため
import os
from IPython import get_ipython

if not os.path.exists('/content/ComfyUI'):
    get_ipython().system('git clone --depth 1 https://github.com/Comfy-Org/ComfyUI /content/ComfyUI')
get_ipython().system('pip install -q -r /content/ComfyUI/requirements.txt')
print('導入が終わりました')

In [ ]:
# 【起動と見張り】ComfyUIを起動し、このセルの下に画面を表示します
# このセルは動き続けます（止めないでください）。生成していない時間が続くと自動で切断します
import socket, subprocess, time, requests
from google.colab import output, runtime, drive

log = open('/content/comfyui.log', 'w')
proc = subprocess.Popen(
    ['python', 'main.py',
     '--offline',                      # 外部との通信と有料API部品を止める
     '--listen', '127.0.0.1', '--port', '8188',
     '--models-directory', MODELS,
     '--output-directory', OUTPUT,
     '--user-directory', USERDIR],
    cwd='/content/ComfyUI', stdout=log, stderr=subprocess.STDOUT)

print('ComfyUIを起動中…（1〜3分）')
for _ in range(600):
    if proc.poll() is not None:
        break
    with socket.socket() as s:
        if s.connect_ex(('127.0.0.1', 8188)) == 0:
            break
    time.sleep(1)

def stop_runtime(reason):
    print(reason + ' → Driveへの書き込みを終えてから切断します')
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    runtime.unassign()

if proc.poll() is not None:
    print('起動に失敗しました。ログの最後を表示します：')
    print(open('/content/comfyui.log').read()[-3000:])
else:
    print('起動しました。この下にComfyUIの画面が表示されます。')
    output.serve_kernel_port_as_iframe(8188, height='900')
    start = time.time()
    idle = 0
    try:
        while True:
            time.sleep(60)
            if proc.poll() is not None:
                print('ComfyUIが停止しました。ログ: /content/comfyui.log')
                break
            try:
                q = requests.get('http://127.0.0.1:8188/queue', timeout=10).json()
                busy = len(q.get('queue_running', [])) + len(q.get('queue_pending', [])) > 0
            except Exception:
                busy = True  # 応答がないのは処理中とみなす
            idle = 0 if busy else idle + 1
            hours = (time.time() - start) / 3600
            if idle >= IDLE_MINUTES:
                stop_runtime(f'{IDLE_MINUTES}分間生成がありませんでした')
                break
            if hours >= MAX_HOURS:
                stop_runtime(f'起動から{MAX_HOURS}時間が過ぎました')
                break
    except KeyboardInterrupt:
        print('見張りを止めました。使い終わったら必ず「ランタイム」→「ランタイムを接続解除して削除」を押してください')

## 使い終わったら

生成が止まって30分たつと自動で切断します。すぐ終えたいときは、メニューの「ランタイム」→「ランタイムを接続解除して削除」を押します。
作った動画はGoogle Driveの `ComfyUI_LTX/output` に保存されています。